In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" / "Livestock_Dataset"

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY -

DATA_DIR = BASE_DATA_DIR / "Livestock_Skin"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: c:\Users\MangOS\livestock-diseases-ai
Đang nạp dữ liệu từ: Poultry_Feces
Thư mục Data tồn tại: True
Torch: 2.2.2+cu121
Device: cuda


In [6]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [7]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 5644, 'val': 1206, 'test': 1217, 'total': 8067}
Number of classes: 4
Trọng số phân lớp (Class Weights): tensor([0.8142, 0.8394, 3.5903, 0.7681])


In [8]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [1, 0, 2, 3, 0, 3, 0, 1, 0, 2]


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Device: cuda
Model device: cuda:0
Model created successfully.
Parameters: {'total': 11178564, 'trainable': 2052, 'frozen': 11176512}


In [10]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 4])


In [11]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_cow_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_cow_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 2,052


Epoch 01/10 | Train Loss: 1.2771 | Train Acc: 0.5328 | Val Loss: 0.5338 | Val Acc: 0.8093 | Time: 30s
  --> Best checkpoint saved! (Val Acc: 0.8093)


Epoch 02/10 | Train Loss: 0.7469 | Train Acc: 0.7314 | Val Loss: 0.4105 | Val Acc: 0.8607 | Time: 11s
  --> Best checkpoint saved! (Val Acc: 0.8607)


Epoch 03/10 | Train Loss: 0.6251 | Train Acc: 0.7739 | Val Loss: 0.3496 | Val Acc: 0.8823 | Time: 11s
  --> Best checkpoint saved! (Val Acc: 0.8823)


Epoch 04/10 | Train Loss: 0.5562 | Train Acc: 0.8007 | Val Loss: 0.3365 | Val Acc: 0.8839 | Time: 11s
  --> Best checkpoint saved! (Val Acc: 0.8839)


...

Epoch 06/10 | Train Loss: 0.4975 | Train Acc: 0.8235 | Val Loss: 0.3153 | Val Acc: 0.8897 | Time: 11s


Epoch 07/10 | Train Loss: 0.5005 | Train Acc: 0.8267 | Val Loss: 0.3019 | Val Acc: 0.9013 | Time: 11s
  --> Best checkpoint saved! (Val Acc: 0.9013)


Epoch 08/10 | Train Loss: 0.4847 | Train Acc: 0.8228 | Val Loss: 0.3030 | Val Acc: 0.8988 | Time: 11s


Epoch 09/10 | Train Loss: 0.4770 | Train Acc: 0.8253 | Val Loss: 0.2979 | Val Acc: 0.9005 | Time: 11s


Epoch 10/10 | Train Loss: 0.4857 | Train Acc: 0.8290 | Val Loss: 0.3021 | Val Acc: 0.8997 | Time: 11s
Phase 1 training completed.


In [12]:
phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
    prefix="cow_",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "cow_training_curves_phase1.png",
)
print("Phase 1 evaluation and plotting completed.")


--- Evaluation Results (phase1) ---
Accuracy   : 0.9047
Macro F1   : 0.8646
Weighted F1: 0.9063
Phase 1 evaluation and plotting completed.


In [13]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_cow_phase1_best.pth"),
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11178564, 'trainable': 11178564, 'frozen': 0}


In [14]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_cow_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_cow_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,178,564


Epoch 01/15 | Train Loss: 0.2940 | Train Acc: 0.9026 | Val Loss: 0.1992 | Val Acc: 0.9328 | Time: 30s
  --> Best checkpoint saved! (Val Acc: 0.9328)


Epoch 02/15 | Train Loss: 0.1686 | Train Acc: 0.9417 | Val Loss: 0.0963 | Val Acc: 0.9594 | Time: 29s
  --> Best checkpoint saved! (Val Acc: 0.9594)


Epoch 03/15 | Train Loss: 0.1141 | Train Acc: 0.9589 | Val Loss: 0.1349 | Val Acc: 0.9552 | Time: 30s


Epoch 04/15 | Train Loss: 0.1027 | Train Acc: 0.9646 | Val Loss: 0.0976 | Val Acc: 0.9652 | Time: 30s
  --> Best checkpoint saved! (Val Acc: 0.9652)


...

Epoch 11/15 | Train Loss: 0.0241 | Train Acc: 0.9892 | Val Loss: 0.1125 | Val Acc: 0.9660 | Time: 29s


Epoch 12/15 | Train Loss: 0.0228 | Train Acc: 0.9908 | Val Loss: 0.0847 | Val Acc: 0.9751 | Time: 30s
  --> Best checkpoint saved! (Val Acc: 0.9751)


Epoch 13/15 | Train Loss: 0.0214 | Train Acc: 0.9920 | Val Loss: 0.0898 | Val Acc: 0.9726 | Time: 30s


Epoch 14/15 | Train Loss: 0.0198 | Train Acc: 0.9922 | Val Loss: 0.0874 | Val Acc: 0.9751 | Time: 30s


Epoch 15/15 | Train Loss: 0.0173 | Train Acc: 0.9933 | Val Loss: 0.0883 | Val Acc: 0.9743 | Time: 29s
Phase 2 training completed.


In [15]:
phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
    prefix="cow_",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "cow_training_curves_phase2.png",
)
print("Phase 2 evaluation completed.")


--- Evaluation Results (phase2) ---
Accuracy   : 0.9778
Macro F1   : 0.9775
Weighted F1: 0.9779
Phase 2 evaluation completed.
